In [1]:
import pandas as pd

In [2]:
customers = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_orders_dataset.csv")
products = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_sellers_dataset.csv")
order_payments = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_order_reviews_dataset.csv")
order_items = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_order_items_dataset.csv")
product_category = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/product_category_name_translation.csv")
geolocation = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/raw/olist_geolocation_dataset.csv")

In [3]:
dataset = {
    "customers": customers,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "order_items": order_items,
    "product_category": product_category,
    "geolocation": geolocation
}


for name, df in dataset.items():
    print("\n", "="*40)
    print(name)
    print("="*40)
    print(df.isnull().sum()) 




customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

products
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

sellers
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

order_payments
order_id              

In [4]:
# Handle missing values

# 1. Orders
# Delivery dates can be missing for orders that were not delivered.
# We keep these missing values because they are meaningful.

# 2. Products
# Fill missing product category with "Unknown"
dataset["products"]["product_category_name"] = (
    dataset["products"]["product_category_name"].fillna("Unknown")
)

# Fill missing product name/description lengths and photos with 0
dataset["products"]["product_name_lenght"] = (
    dataset["products"]["product_name_lenght"].fillna(0)
)

dataset["products"]["product_description_lenght"] = (
    dataset["products"]["product_description_lenght"].fillna(0)
)

dataset["products"]["product_photos_qty"] = (
    dataset["products"]["product_photos_qty"].fillna(0)
)

# Fill the 2 missing physical measurements with the median
for column in [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]:
    dataset["products"][column] = (
        dataset["products"][column].fillna(
            dataset["products"][column].median()
        )
    )

# 3. Order Reviews
# Missing comments mean the customer did not provide a comment.
dataset["order_reviews"]["review_comment_title"] = (
    dataset["order_reviews"]["review_comment_title"].fillna("No comment")
)

dataset["order_reviews"]["review_comment_message"] = (
    dataset["order_reviews"]["review_comment_message"].fillna("No comment")
)

print("Missing values handled successfully!")

Missing values handled successfully!


In [5]:
# Remove duplicate rows from all datasets

for name, df in dataset.items():
    
    # Count duplicates before removing
    duplicate_count = df.duplicated().sum()
    
    print("\n", "=" * 40)
    print(name)
    print("=" * 40)
    print("Duplicates found:", duplicate_count)
    
    # Remove duplicate rows
    dataset[name] = df.drop_duplicates().reset_index(drop=True)
    
    # Show remaining rows
    print("Rows after removing duplicates:", len(dataset[name]))


customers
Duplicates found: 0
Rows after removing duplicates: 99441

orders
Duplicates found: 0
Rows after removing duplicates: 99441

products
Duplicates found: 0
Rows after removing duplicates: 32951

sellers
Duplicates found: 0
Rows after removing duplicates: 3095

order_payments
Duplicates found: 0
Rows after removing duplicates: 103886

order_reviews
Duplicates found: 0
Rows after removing duplicates: 99224

order_items
Duplicates found: 0
Rows after removing duplicates: 112650

product_category
Duplicates found: 0
Rows after removing duplicates: 71

geolocation
Duplicates found: 261831
Rows after removing duplicates: 738332


In [6]:
# Correct data types

# Orders
dataset["orders"]["order_purchase_timestamp"] = pd.to_datetime(
    dataset["orders"]["order_purchase_timestamp"]
)

dataset["orders"]["order_approved_at"] = pd.to_datetime(
    dataset["orders"]["order_approved_at"]
)

dataset["orders"]["order_delivered_carrier_date"] = pd.to_datetime(
    dataset["orders"]["order_delivered_carrier_date"]
)

dataset["orders"]["order_delivered_customer_date"] = pd.to_datetime(
    dataset["orders"]["order_delivered_customer_date"]
)

dataset["orders"]["order_estimated_delivery_date"] = pd.to_datetime(
    dataset["orders"]["order_estimated_delivery_date"]
)

# Order Items
dataset["order_items"]["shipping_limit_date"] = pd.to_datetime(
    dataset["order_items"]["shipping_limit_date"]
)

# Order Reviews
dataset["order_reviews"]["review_creation_date"] = pd.to_datetime(
    dataset["order_reviews"]["review_creation_date"]
)

dataset["order_reviews"]["review_answer_timestamp"] = pd.to_datetime(
    dataset["order_reviews"]["review_answer_timestamp"]
)

print("Data types corrected successfully!")

Data types corrected successfully!


In [7]:
# Check date/time columns

print(dataset["orders"].dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [8]:
# Standardize categorical values

categorical_columns = {
    "customers": [
        "customer_city",
        "customer_state"
    ],
    
    "orders": [
        "order_status"
    ],
    
    "sellers": [
        "seller_city",
        "seller_state"
    ],
    
    "order_payments": [
        "payment_type"
    ],
    
    "product_category": [
        "product_category_name",
        "product_category_name_english"
    ]
}

for dataset_name, columns in categorical_columns.items():
    
    for column in columns:
        
        dataset[dataset_name][column] = (
            dataset[dataset_name][column]
            .astype(str)
            .str.strip()
            .str.lower()
        )

print("Categorical values standardized successfully!")

Categorical values standardized successfully!


In [9]:
# Check for invalid prices

print("Invalid prices:")
print(dataset["order_items"][
    dataset["order_items"]["price"] <= 0
])

# Check for invalid freight values

print("Invalid freight values:")
print(dataset["order_items"][
    dataset["order_items"]["freight_value"] < 0
])

# Check for invalid review scores

print("Invalid review scores:")
print(dataset["order_reviews"][
    ~dataset["order_reviews"]["review_score"].between(1, 5)
])

# Check for invalid product measurements

print("Invalid product measurements:")

print(
    dataset["products"][
        (dataset["products"]["product_weight_g"] <= 0) |
        (dataset["products"]["product_length_cm"] <= 0) |
        (dataset["products"]["product_height_cm"] <= 0) |
        (dataset["products"]["product_width_cm"] <= 0)
    ]
)

Invalid prices:
Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []
Invalid freight values:
Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []
Invalid review scores:
Empty DataFrame
Columns: [review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]
Index: []
Invalid product measurements:
                             product_id product_category_name  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho   
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho   
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho   
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho   

       product_name_lenght  product_description_lenght  product_photos_qty  \
9769                  51.0                       529.0                 1.0   
13683  

In [10]:
# Rename columns where required

dataset["products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    },
    inplace=True
)

print("Columns renamed successfully!")

Columns renamed successfully!


In [11]:
print(dataset["products"].columns)

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')


In [12]:
for name, df in dataset.items():
    print("\n", "=" * 40)
    print(name)
    print("=" * 40)
    print("Rows:", len(df))
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


customers
Rows: 99441
Missing values: 0
Duplicate rows: 0

orders
Rows: 99441
Missing values: 4908
Duplicate rows: 0

products
Rows: 32951
Missing values: 0
Duplicate rows: 0

sellers
Rows: 3095
Missing values: 0
Duplicate rows: 0

order_payments
Rows: 103886
Missing values: 0
Duplicate rows: 0

order_reviews
Rows: 99224
Missing values: 0
Duplicate rows: 0

order_items
Rows: 112650
Missing values: 0
Duplicate rows: 0

product_category
Rows: 71
Missing values: 0
Duplicate rows: 0

geolocation
Rows: 738332
Missing values: 0
Duplicate rows: 0


In [14]:
import os

# Folder where cleaned data will be saved
cleaned_folder = r"C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned"

# Save each cleaned DataFrame
for name, df in dataset.items():
    file_path = os.path.join(cleaned_folder, f"{name}_cleaned.csv")
    df.to_csv(file_path, index=False)
    print(f"{name} saved successfully")
    print(f"Location: {file_path}")

customers saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\customers_cleaned.csv
orders saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\orders_cleaned.csv
products saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\products_cleaned.csv
sellers saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\sellers_cleaned.csv
order_payments saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\order_payments_cleaned.csv
order_reviews saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned\order_reviews_cleaned.csv
order_items saved successfully
Location: C:/Users/Siddarth.b/OneDrive/Docume